[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Adapters and Converters


## What you will be able to do

Store Python values that SQLite has no storage class for, such as `datetime`, `date` and `Decimal`,
by registering an adapter that turns each one into text or a number, and get the same Python types
back with a converter. Switch conversion on with `detect_types`, by declared type or by a name in the
query. Choose between ISO 8601 text and a Unix epoch for times, ask about days and months with
SQLite's date functions, replace the date adapters Python 3.12 deprecated, and read text that is not
UTF-8.


## The idea

### The problem

Every hour in this guide's readings is text, `'2025-03-01T06:00'`, written by `strftime` before it
was stored and read back as a string. A program would rather work with `datetime` objects, which add
a `timedelta`, compare with each other and know their weekday. So every insert starts by turning a
`datetime` into text and every query ends by turning text back into a `datetime`, and one forgotten
conversion stores an hour in another format, which then sorts in the wrong place among the rest.

Handing a `datetime` straight to `execute` appears to work, but only through a built-in adapter that
Python 3.12 deprecated, and the text that adapter writes, `'2025-03-01 06:00:00'`, has a space where
the stored hours have a `T`. A `Decimal`, the type for amounts of money that must not pick up
floating-point error, is refused outright. And on the way out, SQLite returns text, which stays text
unless the connection has been told what to make of it.

### What adapters and converters are

> An **adapter** is a function that turns a Python object into a value SQLite can store, an `int`,
> `float`, `str`, `bytes` or `None`, and `sqlite3.register_adapter(type, function)` makes sqlite3
> call it for every parameter of that type. A **converter** is the reverse: a function that turns a
> stored value, handed to it as `bytes`, back into a Python object, registered under a name with
> `sqlite3.register_converter(name, function)`. sqlite3 calls a converter only on a connection opened
> with **`detect_types`**: `sqlite3.PARSE_DECLTYPES` matches the name against the first word of a
> column's declared type, and `sqlite3.PARSE_COLNAMES` against a name in square brackets in a
> column's alias, as in `AS "hour [datetime]"`. Registrations belong to the `sqlite3` module, so
> they apply to every connection in the program.

### Why it works that way

- **SQLite has five storage classes, and Python has many more types.** A `datetime`, a `date` or a
  `Decimal` has to become text, a number or bytes to be stored, and the program decides which, once,
  in an adapter.
- **An adapter is found by a parameter's exact type.** An adapter for `datetime` does not cover
  `date`, although a `datetime` is a kind of `date`, so every type is registered on its own. A class
  of your own can instead define a `__conform__` method, which sqlite3 calls to adapt it.
- **A converter always receives bytes.** Whatever the storage class, a converter is handed the
  value's text as `bytes`, such as `b'2025-12-01T09:30'`, and decodes it itself. `NULL` never
  reaches a converter, and arrives as `None`.
- **Converters run only when a connection asks for them.** `detect_types` turns the lookup on, by
  declared type, by a name in the query, or both, and without it every stored time comes back as a
  `str`. When both are on, a name in the query wins.
- **Registrations are global.** An adapter or converter registered anywhere applies to every
  connection in the program, so a library or a test that registers one changes what everything else
  stores and receives.
- **The built-in date adapters and converters are deprecated.** Since Python 3.12, binding a `date`
  or `datetime` with no adapter of your own, or converting a column declared `DATE` or `TIMESTAMP`
  with no converter of your own, still works but emits a `DeprecationWarning`, and the Python
  documentation gives recipes to replace them.

### Where this shows up

psycopg, in the **asyncpg and psycopg3, Deep Dive** guide, adapts `date`, `datetime` and `Decimal`
itself, since PostgreSQL has real types for them, and that guide's **Types and Adaptation** notebook
shows the objects that still need a rule. The **Column Types** notebook in the **SQLAlchemy, Deep
Dive** guide shows an ORM storing a `datetime` in SQLite as text, and the precision that can lose.
Text that is not valid UTF-8 is the subject of the **Encodings** notebook in the **Files, Paths and
Formats** guide. And the **DuckDB, Deep Dive** guide works with an SQL that has real date and
timestamp types, where SQLite has functions that read text.

### What this notebook covers

- An adapter for `datetime`, registered once for the whole program
- A converter, switched on by `PARSE_DECLTYPES`
- `PARSE_COLNAMES`, for any column or expression
- ISO 8601 text or a Unix epoch, and converting between them
- SQLite's date functions, for days, months and hours of the day
- When to use `PARSE_DECLTYPES`, `PARSE_COLNAMES`, or neither
- A week of daily extremes, returned as `date` objects
- Six errors: a `Decimal` with no adapter, the deprecated default date adapter, text that is not
  UTF-8, hours stored in two formats, money summed in SQL, and an epoch an hour out

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3
from datetime import datetime

def adapt_datetime(value):
    return value.isoformat(timespec="minutes")

def convert_datetime(value):
    return datetime.fromisoformat(value.decode())

sqlite3.register_adapter(datetime, adapt_datetime)
sqlite3.register_converter("datetime", convert_datetime)

conn = sqlite3.connect(":memory:", detect_types=sqlite3.PARSE_DECLTYPES)
conn.execute("CREATE TABLE visits (station TEXT, arrived DATETIME, departed DATETIME)")
conn.execute("INSERT INTO visits VALUES (?, ?, ?)",
             ("Kirkenes", datetime(2025, 12, 1, 9, 30), datetime(2025, 12, 1, 14, 15)))

print(conn.execute("SELECT arrived, typeof(arrived) FROM visits").fetchone())
arrived, departed = conn.execute("SELECT arrived, departed FROM visits").fetchone()
print(type(arrived).__name__, "on site for", departed - arrived)
conn.close()
```

```
(datetime.datetime(2025, 12, 1, 9, 30), 'text')
datetime on site for 4:45:00
```

Two registrations, and a `datetime` went in and came back out as a `datetime`. In between, SQLite
held text, as `typeof` shows: the adapter wrote it, and the converter, which `detect_types` switched
on, read it back. Subtracting one time from the other then gave a `timedelta`, with no conversion
written anywhere near the queries.


## Setup

Eight imports, and the stations' year, built into the two tables the **Tables and Queries** notebook
designed.

- `sqlite3` builds the database, runs every statement, and holds the registered adapters and
  converters
- `date`, `datetime`, `timedelta` and `timezone`, from `datetime`, are the times this notebook
  stores, adds and converts
- `Decimal`, from `decimal`, is the exact number that needs an adapter of its own, in Common errors
- `ZoneInfo`, from `zoneinfo`, gives Oslo's time zone, in Common errors
- `warnings` catches the deprecation warning in Common errors
- `math` makes the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end


In [1]:
import math
import shutil
import sqlite3
import warnings
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path
from zoneinfo import ZoneInfo

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### An adapter for datetime

`sqlite3.register_adapter` takes a Python type and a function, and from then on sqlite3 calls that
function on every parameter of that type, on every connection the program opens. This adapter writes
a `datetime` in the form the readings' hours already have, ISO 8601 to the minute. The first readings
from Kirkenes then go in as `datetime` objects:


In [2]:
def adapt_datetime(value):
    """A datetime as ISO 8601 text to the minute, the form every hour in readings has."""
    return value.isoformat(timespec="minutes")


sqlite3.register_adapter(datetime, adapt_datetime)

conn = sqlite3.connect(DATABASE)
station_ids = dict(conn.execute("SELECT name, id FROM stations"))

first_hour = datetime(2025, 12, 1, 0, 0)
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 [(station_ids["Kirkenes"], first_hour + timedelta(hours=n), celsius)
                  for n, celsius in enumerate([-7.4, -7.9, -8.3, -8.1, -8.6, -8.8])])
conn.commit()

stored = conn.execute("SELECT hour, typeof(hour) FROM readings WHERE station_id = ? ORDER BY hour LIMIT 2",
                      (station_ids["Kirkenes"],)).fetchall()
print("stored:", stored)
print("read back as:", type(stored[0][0]).__name__)


stored: [('2025-12-01T00:00', 'text'), ('2025-12-01T01:00', 'text')]
read back as: str


Six `datetime` objects went in as the same text the Svalbard readings have, so they sort and compare
with every other hour in the table. Coming out, they are still text, since nothing has asked for a
conversion yet.

### A converter, and PARSE_DECLTYPES

A converter is registered under a name, and `detect_types` on a connection decides where sqlite3
looks for that name. With `sqlite3.PARSE_DECLTYPES`, it reads the first word of the declared type of
the column a value came from, `DATETIME` below, and finds the converter registered under that word,
in any case. Here a table of maintenance visits declares its times as `DATETIME`:


In [3]:
def convert_datetime(value):
    """ISO 8601 text, which a converter receives as bytes, back into a datetime."""
    return datetime.fromisoformat(value.decode())


sqlite3.register_converter("datetime", convert_datetime)
conn.close()
conn = sqlite3.connect(DATABASE, detect_types=sqlite3.PARSE_DECLTYPES | sqlite3.PARSE_COLNAMES)

conn.execute("""
    CREATE TABLE visits (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL,
        arrived    DATETIME NOT NULL,
        departed   DATETIME
    )
""")
conn.executemany("INSERT INTO visits (station_id, arrived, departed) VALUES (?, ?, ?)", [
    (station_ids["Kirkenes"], datetime(2025, 12, 1, 9, 30), datetime(2025, 12, 1, 14, 15)),
    (station_ids["Svalbard"], datetime(2025, 3, 3, 8, 0), None),
])
conn.commit()

for arrived, departed in conn.execute("SELECT arrived, departed FROM visits ORDER BY id"):
    on_site = departed - arrived if departed else "still there"
    print(type(arrived).__name__, arrived, "to", departed, "->", on_site)
print("MIN(arrived):", repr(conn.execute("SELECT MIN(arrived) FROM visits").fetchone()[0]))


datetime 2025-12-01 09:30:00 to 2025-12-01 14:15:00 -> 4:45:00
datetime 2025-03-03 08:00:00 to None -> still there
MIN(arrived): '2025-03-03T08:00'


Both arrivals came back as `datetime` objects, so subtracting them gave a `timedelta`. The missing
departure never reached the converter and arrived as `None`. `MIN(arrived)` came back as text,
because an expression has no declared type for `PARSE_DECLTYPES` to read.

`DATETIME` is not one of the type names a `STRICT` table allows, as the **Type Affinity** notebook
showed, so this table is a flexible one. SQLite gives the column NUMERIC affinity, and ISO 8601 text,
which is not a number, stays text. The connection has both flags set, which the next section uses.

### PARSE_COLNAMES, for any column or expression

With `sqlite3.PARSE_COLNAMES`, the converter's name comes from the query: square brackets after a
column's alias name the converter for that column. That works for an expression, and for a column
declared `TEXT`, as `readings.hour` is. sqlite3 removes the bracketed name from the column's name in
`description`:


In [4]:
cursor = conn.execute("""
    SELECT MIN(hour) AS "first [datetime]", MAX(hour) AS "last [datetime]"
    FROM readings
    WHERE station_id = ?
""", (station_ids["Kirkenes"],))
first, last = cursor.fetchone()

print("columns:", [column[0] for column in cursor.description])
print(type(first).__name__, first, "to", last, "spans", last - first)


columns: ['first', 'last']
datetime 2025-12-01 00:00:00 to 2025-12-01 05:00:00 spans 5:00:00


The same converter served both flags. When a column has both a declared type and a bracketed name,
the name in the query wins, so a query can always choose its own conversion.

### ISO 8601 text or a Unix epoch

A time can be stored as text or as a number. ISO 8601 text such as `'2025-03-01T06:00'` reads
plainly, sorts in time order, and suits SQLite's date functions. A Unix epoch, the whole seconds
since the start of 1970 in UTC, is a single integer, with no format to disagree about. Either
converts to the other, in Python and in SQLite, as long as the time zone is stated:


In [5]:
hour = datetime(2025, 3, 1, 6, 0, tzinfo=timezone.utc)
epoch = int(hour.timestamp())
print("Python:", hour, "->", epoch, "->", datetime.fromtimestamp(epoch, timezone.utc))

as_epoch, as_text = conn.execute("SELECT strftime('%s', '2025-03-01T06:00'), datetime(?, 'unixepoch')", (epoch,)).fetchone()
print("SQLite:", repr(as_epoch), "and", repr(as_text))


Python: 2025-03-01 06:00:00+00:00 -> 1740808800 -> 2025-03-01 06:00:00+00:00
SQLite: '1740808800' and '2025-03-01 06:00:00'


SQLite reads a time with no zone as UTC, so `strftime('%s', ...)` gave the same epoch that Python
computed from a UTC `datetime`. Two details: `strftime('%s')` returns the epoch as text, which a
column with INTEGER affinity turns back into a number, and SQLite's `datetime()` writes a space where
the stored hours have a `T`. SQLite 3.38.0 added `unixepoch()`, a shorter way to an integer epoch.

| Store | When | Why |
|---|---|---|
| ISO 8601 text, `'2025-03-01T06:00'` | most tables, and any time a person will read | it sorts in time order as text, suits `date()` and `strftime()`, and needs no decoding to read |
| a Unix epoch, `1740808800` | times from a system that already counts seconds, or durations worked out in SQL | one integer, with no format to disagree about, and a duration is a subtraction |

The default is ISO 8601 text, in one format for every row, in UTC.

### SQLite's date functions

SQLite has no date type, but its date functions read ISO 8601 text: `date()` and `datetime()` return
a date or a date and time, `strftime()` formats any part, and `julianday()` returns a day number,
whose differences are durations in days. A question about a month is best asked as a range of text,
which the **Indexes and Query Plans** notebook shows an index can serve, and a question about a part
of every time, such as the hour of the day, with `strftime()`:


In [6]:
in_march = conn.execute("""
    SELECT COUNT(*) FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.hour >= ? AND r.hour < ?
""", ("Svalbard", datetime(2025, 3, 1), datetime(2025, 4, 1))).fetchone()[0]
print("Svalbard's hours in March:", in_march)

coldest_hours = conn.execute("""
    SELECT strftime('%H', r.hour) AS hour_of_day, ROUND(AVG(r.celsius), 1) AS mean
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.hour >= ? AND r.hour < ?
    GROUP BY hour_of_day
    ORDER BY AVG(r.celsius)
    LIMIT 3
""", ("Svalbard", datetime(2025, 1, 1), datetime(2025, 2, 1))).fetchall()
print("coldest hours of the day in January:", coldest_hours)

print("a month on, and a day and a quarter:", conn.execute("""
    SELECT date('2025-03-01T06:00', '+1 month'), julianday('2025-03-02T06:00') - julianday('2025-03-01T00:00')
""").fetchone())


Svalbard's hours in March: 744
coldest hours of the day in January: [('03', -16.4), ('04', -16.3), ('02', -16.3)]
a month on, and a day and a quarter: ('2025-04-01', 1.25)


The `datetime` parameters became the ends of a range of text, from `'2025-03-01T00:00'` up to but not
including `'2025-04-01T00:00'`, which is every hour of March. `strftime('%H', ...)` took the hour of
the day from every reading, so `GROUP BY` could put all 31 of January's 03:00 readings together. The
last line moved a date by a month, and measured the time between two hours in days.

### DECLTYPES, COLNAMES, or neither

`detect_types` can look for converters in two places, or nowhere:

| Write | When | Why |
|---|---|---|
| `PARSE_DECLTYPES` | a flexible table whose columns are declared with a converter's name, such as `DATETIME` | every query converts those columns, with nothing written in the query |
| `PARSE_COLNAMES` | a `STRICT` table, an expression such as `MIN(hour)`, or one query that needs a conversion | the alias names the converter, for any column or expression |
| neither, converting in Python | a handful of values, or code that must not depend on what the program has registered | nothing global changes what a query returns |

The default is both flags together, as this notebook's connection has, with conversions named in
queries wherever a table is `STRICT`, since a `STRICT` table's declared types can only be `INT`,
`INTEGER`, `REAL`, `TEXT`, `BLOB` or `ANY`.

### A week of daily extremes, as dates

The pieces of this notebook in one job: every day's coldest and warmest reading at a station, for a
number of days from a given day, with each day returned as a `date` object. The range comes in as two
`datetime` parameters, SQLite's `date()` groups the readings by day, and a converter registered as
`date` turns each day back into a `date`, which Python then formats with its weekday:


In [7]:
def convert_date(value):
    """ISO 8601 date text, received as bytes, back into a date."""
    return date.fromisoformat(value.decode())


sqlite3.register_converter("date", convert_date)


def daily_extremes(conn, station, first_day, days):
    """Every day's coldest and warmest reading at a station, for a number of days from first_day."""
    return conn.execute("""
        SELECT date(r.hour) AS "day [date]", MIN(r.celsius) AS coldest, MAX(r.celsius) AS warmest,
               COUNT(r.celsius) AS readings
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ? AND r.hour >= ? AND r.hour < ?
        GROUP BY date(r.hour)
        ORDER BY date(r.hour)
    """, (station, first_day, first_day + timedelta(days=days))).fetchall()


for day, coldest, warmest, readings in daily_extremes(conn, "Svalbard", datetime(2025, 2, 28), 4):
    if readings:
        print(f"{day:%a %d %b}  {coldest:6.1f} to {warmest:5.1f}  from {readings} readings")
    else:
        print(f"{day:%a %d %b}  no readings")


Fri 28 Feb   -15.0 to  -8.1  from 24 readings
Sat 01 Mar   -14.7 to  -8.0  from 24 readings
Sun 02 Mar  no readings
Mon 03 Mar   -14.2 to  -7.2  from 24 readings


Four days, and 2 March, when Svalbard sent nothing, is there with no extremes, since the day's hours
exist with `NULL` readings. `day` is a real `date`, so `f"{day:%a %d %b}"` could name its weekday.
Registering a converter called `date` also replaced sqlite3's deprecated built-in converter of that
name, so no warning appeared.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `first_day` and `first_day + timedelta(days=days)` as parameters | datetimes written as ISO 8601 text by an adapter | An adapter for datetime |
| `convert_date` receiving bytes | a converter registered under a name | A converter, and PARSE_DECLTYPES |
| `date(r.hour) AS "day [date]"` | a converter named in an alias, for an expression | PARSE_COLNAMES, for any column or expression |
| `r.hour >= ? AND r.hour < ?` | a range of ISO 8601 text, in time order | SQLite's date functions |
| `GROUP BY date(r.hour)` | SQLite's `date()` reading the stored text | SQLite's date functions |
| `detect_types` with both flags | a name in the query winning over a declared type | DECLTYPES, COLNAMES, or neither |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/09-adapters-and-converters-solutions.ipynb).

**1.** Insert a Kirkenes reading of -9.4 for 06:00 on 1 December, passing the hour as a `datetime`,
and print the stored hour with its `typeof`.


In [8]:
# your code here


**2.** Fetch the hour of Oslo's warmest reading as a `datetime`, using `PARSE_COLNAMES`, and print it
with the name of its weekday.


In [9]:
# your code here


**3.** Register an adapter that stores a `timedelta` as whole seconds and a converter named
`duration` that turns them back, then store and read back a visit's length in a column declared
`DURATION`.


In [10]:
# your code here


**4.** Count Bergen's readings in every month of 2025 with `strftime`, and print the first three
months.


In [11]:
# your code here


**5.** Find the hour of Svalbard's coldest reading, turn it into a Unix epoch in SQL, and turn that
epoch back into a `datetime` in UTC in Python.


In [12]:
# your code here


**6.** Give a class `Coordinates`, with a `latitude` and a `longitude`, a `__conform__` method that
stores it as the text `'latitude;longitude'`, and show what `SELECT ?` returns for Oslo at 59.91,
10.75.


In [13]:
# your code here


## Common errors

### sqlite3.ProgrammingError: Error binding parameter 2: type 'decimal.Decimal' is not supported


In [14]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS repairs (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL,
        cost       TEXT NOT NULL,
        day        TEXT NOT NULL
    ) STRICT
""")
conn.execute("INSERT INTO repairs (station_id, cost, day) VALUES (?, ?, ?)",
             (station_ids["Kirkenes"], Decimal("1249.90"), "2025-12-01"))


ProgrammingError: Error binding parameter 2: type 'decimal.Decimal' is not supported

The second parameter, the cost, is a `Decimal`, and sqlite3 has no adapter for it, so binding
stopped there. A `Decimal` holds an amount exactly, and turning it into a `float` would give up
exactly that, so store it as text: an adapter that writes `str(value)`, and a converter that reads
the text back into a `Decimal`, named in the query:


In [15]:
def adapt_decimal(value):
    """A Decimal as exact text, so no floating-point rounding ever touches it."""
    return str(value)


def convert_decimal(value):
    """Exact decimal text, received as bytes, back into a Decimal."""
    return Decimal(value.decode())


sqlite3.register_adapter(Decimal, adapt_decimal)
sqlite3.register_converter("decimal", convert_decimal)

conn.execute("INSERT INTO repairs (station_id, cost, day) VALUES (?, ?, ?)",
             (station_ids["Kirkenes"], Decimal("1249.90"), "2025-12-01"))
conn.commit()
print(conn.execute('SELECT cost AS "cost [decimal]", typeof(cost) FROM repairs').fetchone())


(Decimal('1249.90'), 'text')


### DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes


In [16]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    conn.execute("INSERT INTO repairs (station_id, cost, day) VALUES (?, ?, ?)",
                 (station_ids["Kirkenes"], Decimal("89.90"), date(2025, 12, 8)))
conn.commit()

for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")
print(conn.execute("SELECT day, typeof(day) FROM repairs ORDER BY id DESC").fetchone())


('2025-12-08', 'text')


The insert worked, and the warning says it will not work forever. This notebook registered an
adapter for `datetime`, and an adapter is found by a parameter's exact type, so the `date` went
through sqlite3's own adapter, deprecated since Python 3.12. It still writes ISO 8601, but a
program's storage format should not rest on a default that is going away. Register the adapter the
Python documentation's recipe gives, which writes the same text, and the warning goes:


In [17]:
sqlite3.register_adapter(date, date.isoformat)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    stored = conn.execute("SELECT ?", (date(2025, 12, 8),)).fetchone()[0]
print("stored as:", repr(stored), "with", len(caught), "warnings")


stored as: '2025-12-08' with 0 warnings


### sqlite3.OperationalError: Could not decode to UTF-8 column 'name' with text 'Troms�'


In [18]:
conn.execute("CREATE TABLE IF NOT EXISTS old_names (name TEXT NOT NULL)")
conn.execute("INSERT INTO old_names VALUES (?)", ("Tromsø",))
conn.execute("INSERT INTO old_names VALUES (CAST(? AS TEXT))", (b"Troms\xf8",))   # as an older export wrote it
conn.commit()

print(conn.execute("SELECT name FROM old_names").fetchall())


OperationalError: Could not decode to UTF-8 column 'name' with text 'Troms�'

sqlite3 decodes every TEXT value as UTF-8, and the second name holds the byte `F8`, which is `ø` in
Latin-1 but is not valid UTF-8 on its own, so reading the column failed, the good row with it. The
**Encodings** notebook in the **Files, Paths and Formats** guide explains the bytes. A connection's
`text_factory` decides how its TEXT values are decoded: this one tries UTF-8 and falls back to
Latin-1, reads the names once, writes them back as proper UTF-8, and then puts the default back:


In [19]:
def decode_text(data):
    """UTF-8 when the bytes are valid UTF-8, and Latin-1, which accepts any byte, when they are not."""
    try:
        return data.decode("utf-8")
    except UnicodeDecodeError:
        return data.decode("latin-1")


conn.text_factory = decode_text
names = conn.execute("SELECT rowid, name FROM old_names").fetchall()
conn.executemany("UPDATE old_names SET name = ? WHERE rowid = ?", [(name, rowid) for rowid, name in names])
conn.commit()
conn.text_factory = str

print(conn.execute("SELECT name FROM old_names").fetchall())


[('Tromsø',), ('Tromsø',)]


### No error, and a reading missing from its own day: hours stored in two formats


In [20]:
older_rows = [(station_ids["Kirkenes"], f"2025-12-01 {hour:02d}:00:00", celsius)   # as the default adapter wrote hours
              for hour, celsius in [(6, -9.0), (7, -9.2), (8, -9.1)]]
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)", older_rows)
conn.commit()

on_the_day = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND hour >= ? AND hour < ?",
                          (station_ids["Kirkenes"], datetime(2025, 12, 1), datetime(2025, 12, 2))).fetchone()[0]
print("readings on 1 December:", on_the_day, "of", len(older_rows) + 6)
first_four = conn.execute("SELECT hour FROM readings WHERE station_id = ? ORDER BY hour LIMIT 4", (station_ids["Kirkenes"],))
print(first_four.fetchall())


readings on 1 December: 6 of 9
[('2025-12-01 06:00:00',), ('2025-12-01 07:00:00',), ('2025-12-01 08:00:00',), ('2025-12-01T00:00',)]


Three readings were written by an older program that relied on the default datetime adapter, which
puts a space between the date and the time. A space sorts before `T`, so `'2025-12-01 06:00:00'` is
less than `'2025-12-01T00:00'`: those three rows fell outside the day's range, and sorted before the
day's first hour. Nothing raised, since both are valid text. Keep every row in one format, and bring
the old rows into it with SQLite's own `strftime`, which reads both:


In [21]:
conn.execute("UPDATE readings SET hour = strftime('%Y-%m-%dT%H:%M', hour) WHERE hour LIKE '% %'")
conn.commit()

on_the_day = conn.execute("SELECT COUNT(*) FROM readings WHERE station_id = ? AND hour >= ? AND hour < ?",
                          (station_ids["Kirkenes"], datetime(2025, 12, 1), datetime(2025, 12, 2))).fetchone()[0]
print("readings on 1 December:", on_the_day)
first_four = conn.execute("SELECT hour FROM readings WHERE station_id = ? ORDER BY hour LIMIT 4", (station_ids["Kirkenes"],))
print(first_four.fetchall())


readings on 1 December: 9
[('2025-12-01T00:00',), ('2025-12-01T01:00',), ('2025-12-01T02:00',), ('2025-12-01T03:00',)]


### No error, and 1339.8000000000002 kroner: money summed in SQL


In [22]:
total = conn.execute("SELECT SUM(cost) FROM repairs WHERE station_id = ?", (station_ids["Kirkenes"],)).fetchone()[0]

print("SUM(cost):", repr(total))


SUM(cost): 1339.8000000000002


The costs were stored as exact text, 1249.90 and 89.90, but `SUM` works in floating point, so it
turned the text into binary fractions, which cannot hold 1249.90 exactly, and added those. The answer
is off in its last digits, and a report that rounds it would hide that, until two totals that should
match do not. Add exact amounts in Python, as `Decimal` objects. A table that must total money in
SQL stores whole øre as integers instead, which `SUM` adds exactly:


In [23]:
costs = [cost for (cost,) in conn.execute('SELECT cost AS "cost [decimal]" FROM repairs WHERE station_id = ?',
                                          (station_ids["Kirkenes"],))]
print("in Python:", costs, "->", sum(costs))


in Python: [Decimal('1249.90'), Decimal('89.90')] -> 1339.80


### No error, and an hour out: an epoch made from a time with no zone


In [24]:
naive = datetime(2025, 3, 1, 6, 0)          # an hour with no time zone, as every stored hour is

# naive.timestamp() uses the time zone of the machine that runs it, such as these two:
on_a_laptop_in_oslo = int(naive.replace(tzinfo=ZoneInfo("Europe/Oslo")).timestamp())
on_a_server_in_utc = int(naive.replace(tzinfo=timezone.utc).timestamp())

print("epochs:", on_a_laptop_in_oslo, on_a_server_in_utc, "->", on_a_server_in_utc - on_a_laptop_in_oslo, "seconds apart")
print("the laptop's epoch, read back in UTC:", datetime.fromtimestamp(on_a_laptop_in_oslo, timezone.utc))


epochs: 1740805200 1740808800 -> 3600 seconds apart
the laptop's epoch, read back in UTC: 2025-03-01 05:00:00+00:00


`timestamp()` on a `datetime` with no time zone treats it as local time, so the same hour becomes an
epoch that depends on where the program happened to run: an hour apart between a laptop in Oslo in
March and a server in UTC. Read back in UTC, the laptop's reading moved to 05:00, with no error
anywhere. The epoch recipe in the Python documentation, `fromtimestamp(int(val))`, has the same
blind spot in the other direction, returning the machine's local time. State the zone on the way in
and on the way out:


In [25]:
def adapt_to_epoch(value):
    """A datetime as whole seconds since 1970 in UTC, treating a time with no zone as UTC."""
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    return int(value.timestamp())


def convert_from_epoch(value):
    """Whole seconds since 1970, received as bytes, back into a datetime in UTC."""
    return datetime.fromtimestamp(int(value), timezone.utc)


epoch = adapt_to_epoch(naive)
print(epoch, "->", convert_from_epoch(str(epoch).encode()))
conn.close()


1740808800 -> 2025-03-01 06:00:00+00:00


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [26]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- An adapter, registered with `sqlite3.register_adapter`, turns a Python type SQLite cannot store
  into text, a number or bytes whenever a parameter of that exact type is bound.
- A converter, registered with `sqlite3.register_converter`, turns a stored value's bytes back into a
  Python object, and runs only on a connection opened with `detect_types`. `NULL` never reaches one.
- `PARSE_DECLTYPES` finds a converter by a column's declared type, which a `STRICT` table cannot use,
  and `PARSE_COLNAMES` by a name in brackets in an alias, for any column or expression, winning when
  both are set.
- Registrations are global to the `sqlite3` module, and the built-in date adapters and converters,
  deprecated since Python 3.12, are replaced by registering your own.
- Store times as ISO 8601 text in one format, or as a Unix epoch, always in UTC, and ask about days,
  months and hours with a range of text and SQLite's `date()` and `strftime()`.
- A `Decimal` stored as text stays exact until SQL does arithmetic on it, and `text_factory` decides
  how TEXT that is not UTF-8 is read.


## What is next

The **Transactions** notebook turns to what happens between a statement and the file: when changes
become permanent with `commit`, how `rollback` takes them back, and why rows written without a commit
are gone when the file is opened again.


---

&#8592; **Previous:** [Type Affinity](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/08-type-affinity.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/10-transactions.ipynb) &#8594;
